In [7]:
import pandas as pd
import math
import numpy as np
import csv

# -------------------------
# 1. Define overall simulation parameters.
# -------------------------

timesteps = 96                                   # Simulation length, in timesteps
timestep_length_hours = 0.25                     # Timestep length, in hours

base_price = 0.27                                # Average energy price (£)
amplitude = 0.2                                  # Sin wave amplitude, determining price variation

max_charge_percentage = 0.8                      # Allowable maximum charge level for each individual, as a % of total battery capacity
min_charge_percentage = 0.2                      # Allowable minimum charge level for each individual, as a % of total battery capacity

miles_per_kWh = 3                                # How far each EV can travel (in miles), per kWh of energy
kWh_per_mile = 1 / miles_per_kWh                 # kWh of energy expended per mile driven

range_buffer = 1.5                               # Charge level buffer, over and above upcoming trip energy requirement

action_levels = [0.10, 0.20, 0.30, 0.40, 0.50]   # Discrete charge and discharge action increments (charge or discharge 10%, 20%, 30% etc)
use_price_utility = True                         # When True, price effects are included in the charging and discharging utility functions


# -------------------------
# 2. Piecewise energy term 
# -------------------------

# Calculates charging and discharging action energy level related utility scores, based on how attractive post charge energy levels are, relative to the target energy requirement
# E_post is the post-action battery energy level (kWh), after charging or discharging
# E_req is the target charge level (kWh) (see section 6)
# C is battery_capacity_kWh (see section 3.1)
# k_shortfall controls how steeply utility drops when below target (set to 15)
# k_surplus controls how gently utility drops when above target (set to 8)

def piecewise_energy_term(E_post, E_req, C, k_shortfall=15.0, k_surplus=8.0):
    gap_signed = (E_post - E_req) / C                                                     # The normalised difference between the post-action energy level and the target, expressed as a proportion of battery capacity
    dist = abs(gap_signed)                                                                # Takes the absolute value of the signed gap, giving the magnitude of the deviation from the target regardless of direction
    if gap_signed < 0:
        term = np.exp(-k_shortfall * dist)                                                # If gap_signed < 0, there is an energy shortfall, meaning a stronger drop off in utility as shortfall increases
    else:
        term = np.exp(-k_surplus * dist)                                                  # If gap_signed > 0, there is an energy surplus, meaning a gentler drop off in utility as shortfall increases
    return term, gap_signed, dist


# -------------------------
# 3. Function returning per-individual simulation parameters for each individual (through df_sim dataframe)
# -------------------------

def simulation_parameters_for_each_individual(
    base_price, amplitude, betas, individual_id, age_group, gender, segment,
    departure_timestep_home, arrival_timestep_work, departure_timestep_work, arrival_timestep_home,
    trip_distance_out, trip_distance_return,
    avg_speed_out, avg_speed_return, trip_start_out_mins, trip_end_out_mins,
    trip_start_return_mins, trip_end_return_mins
):

# 3.1 Assign EV battery parameters for each individual.
    
    battery_capacity_kWh = np.random.choice([40, 60, 80])              # Randomly set at 40, 60 or 80 kWh
    initial_soc = np.clip(np.random.normal(0.5, 0.15), 0, 1)           # Initial state of charge (SOC) for each individual, randomly selected from a normal distribution with: mean (50%), SD (15%), min (0%), max (100%)
    initial_energy_kWh = battery_capacity_kWh * initial_soc            # Initial energy level (kWh) for each individual 
    max_charge_kWh = battery_capacity_kWh * max_charge_percentage      # Maximum charge level (kWh) for each individual
    min_charge_kWh = battery_capacity_kWh * min_charge_percentage      # Allowable minimum charge level for each individual, as a % of total battery capacity

# 3.2 Assign home and work charge rates for each individual.
    
    home_charger_kw = np.random.choice([3, 7.4, 11])                                 # Randomly set at 3, 7.4 or 11 kW
    work_charger_kw = np.random.choice([7.4, 11, 22])                                # Randomly set at 7.4, 11 or 22 kW
    home_charging_rate_kWh_per_timestep = home_charger_kw * timestep_length_hours    # Home charge rate per timestep in kW/h
    work_charging_rate_kWh_per_timestep = work_charger_kw * timestep_length_hours    # Work charge rate per timestep in kW/h

# 3.3 data imported from the National Travel Survey.
    
    #  The following fields are imported / dervived from the NTS 2023 dataset, commuter data, with timings converted into timesteps where relevant: departure_timestep_home, arrival_timestep_work, departure_timestep_work, arrival_timestep_home, trip_distance_out, trip_distance_return
    #  age_group, gender, segment, trip_start_out_mins, trip_end_out_mins, trip_start_return_mins, trip_end_return_mins
    #  See section 11.

# 3.4 Trip energy requirements for each individual.
    
    energy_req_home_to_work_kWh = trip_distance_out * kWh_per_mile                   # Energy required (kWh) for the home to work trips, for each individual
    energy_req_work_to_home_kWh = trip_distance_return * kWh_per_mile                # Energy required (kWh) for the work to home trip, for each individual

# 3.5 Trip energy consumption for each individual.

    miles_per_timestep_out = avg_speed_out * timestep_length_hours                            # Miles driven per timestep for the home to work trip
    miles_per_timestep_return = avg_speed_return * timestep_length_hours                      # Miles driven per timestep for the work to home trip 
    driving_consumption_per_timestep_out_kWh = miles_per_timestep_out * kWh_per_mile          # Energy (kWh) consumed per timestep for the home to work trip
    driving_consumption_per_timestep_return_kWh = miles_per_timestep_return * kWh_per_mile    # Energy (kWh) consumed per timestep for the work to home trip

# 3.6 betas

    #  These are defined in section 10, then included in the dictionaries in section 11.


# -------------------------
# 4. Charging and discharging choice model - parameter values
# -------------------------

# In timesteps where an EV is not driving or charging, decisions are taken to charge, discharge or do nothing. 
# Choices are in a multinomial logit (MNL) format, meaning that each action has a utility function and actions are then selected randomly across assigned probabilities.
# The action space is discretised, meaning that charging and discharging actions can be taken, charging or discharging between 10% and 50% of battery capacity, in 10% increments.

# 4.1 MNL parameter values, per individual

# Beta values (defined in section 10) are returned for each individual, via the betas dictionary (see section 11), depending on their segment (read from the NTS data).
# These reflect individual attitudes to attributes in each utility function. 
    
    beta0_charge = betas["beta0_charge"]
    beta1_charge = betas["beta1_charge"]
    beta2_charge = betas["beta2_charge"]
    beta0_discharge = betas["beta0_discharge"]
    beta1_discharge = betas["beta1_discharge"]
    beta2_discharge = betas["beta2_discharge"]
    beta0_none = 0.0


# -------------------------
# 5. Determine location for each individual, by timestep
# -------------------------

# A list is created, appending location by timestep, with locations determined by the home and work departure and arrival times provided in the NTS data.

    locations = []
    for timestep in range(timesteps):
        if timestep < departure_timestep_home:
            locations.append("home")
        elif timestep < arrival_timestep_work:
            locations.append("driving_out")
        elif timestep < departure_timestep_work:
            locations.append("work")
        elif timestep < arrival_timestep_home:
            locations.append("driving_return")
        else:
            locations.append("home")


# -------------------------
# 6. Determine target charge level for each individual, by timestep
# -------------------------
    
# Return a target charge level for a charge session, that is equal either to the upcoming energy requirement or maximum charge level.
# Upcoming energy requirement equals the allowable minimum SoC, plus the upcoming journey energy, plus a buffer.

    def target_charge_level(timestep, location):
        if location == "home":
            upcoming_energy_requirement = min_charge_kWh + energy_req_home_to_work_kWh * range_buffer
            return min(max_charge_kWh, upcoming_energy_requirement)
        elif location == "work":
            upcoming_energy_requirement = min_charge_kWh + energy_req_work_to_home_kWh * range_buffer
            return min(max_charge_kWh, upcoming_energy_requirement)
        return None


# -------------------------
# 7. Initiate data field values
# -------------------------

# The following fields are given the following initialising values at the start of the simulation (in many cases initialising at zero):

    accumulated_profit = 0.0                                    # Accumulates profit (£) over each timestep (if negative, this reflects a loss)
    action_duration = 0                                         # Returns the remaining timesteps for an ongoing charging or discharging action
    current_action = None                                       # Defines a selected action that is playing out over multiple timesteps
    charge_session_remaining_kWh = 0.0                          # The remaining energy (kWh) to charge or discharge during a charge session

    current_total_charge_kWh = np.nan                           # The total charge amount (kWh) for any current charging action
    current_total_discharge_kWh = np.nan                        # The total discharge amount (kWh) for any current discharging action

    output = []                                                 # Dataframe including all output data
    battery_energy_level_current_kWh = initial_energy_kWh       # Starting battery energy level (kWh)

    for timestep in range(timesteps):
        location = locations[timestep]                          # Location by timestep is assigned by the For Loop in section 5
        total_minutes = timestep * 15                           # Total minutes elapsed at the start of each timestep
        hour = total_minutes // 60                              # Hours elapsed at the start of each timestep
        minute = total_minutes % 60                             # Minutes past the hour at the start of each timestep
        time_str = f"{int(hour):02d}:{int(minute):02d}"         # Hours and minutes elapsed at the start of each timestep

        price_at_timestep_t = base_price + amplitude * math.sin(2 * math.pi * (timestep / timesteps - 0.25))      # Energy price (£) at each timestep

        Utility_none = np.nan                                   # The utility value of neither charging nor discharging, at a given timestep
        P_none = np.nan                                         # The probability of selecting neither a charging action nor a discharging action, at a given timestep           
        P_charge = np.nan                                       # The probability of selecting a charging action, at a given timestep
        P_discharge = np.nan                                    # The probability of selecting a discharging action, at a given timestep

        utility_charge = {lvl: np.nan for lvl in action_levels}                              # Initialises the charge utility dictionary for each action level
        utility_discharge = {lvl: np.nan for lvl in action_levels}                           # Initialises the discharge utility dictionary for each action level
        prob_charge = {lvl: np.nan for lvl in action_levels}                                 # Initialises the charge probability dictionary for each action level
        prob_discharge = {lvl: np.nan for lvl in action_levels}                              # Initialises the discharge probability dictionary for each action level

        charge_utility_energy_level_by_level = {lvl: np.nan for lvl in action_levels}        # Initialises the energy level component of charge utility dictionary for each action level
        discharge_utility_energy_level_by_level = {lvl: np.nan for lvl in action_levels}     # Initialises the energy level component of discharge utility dictionary for each action level

        distance_after_charge = {lvl: np.nan for lvl in action_levels}                       # Initialises the absolute distance (in % SoC) from required energy level post charge dictionary for each action level
        distance_after_discharge = {lvl: np.nan for lvl in action_levels}                    # Initialises the absolute distance (in % SoC) from required energy level post discharge dictionary for each action level
        shortfall_after_charge = {lvl: np.nan for lvl in action_levels}                      # Initialises the shortfall vs required energy level post charging (in % SoC) dictionary for each action level
        surplus_after_discharge = {lvl: np.nan for lvl in action_levels}                     # Initialises the surplus vs required energy level post discharging (in % SoC) dictionary for each action level
        feasible_charge = {lvl: 0 for lvl in action_levels}                                  # Initialises the charging feasibility dictionary for each action level. Charge action levels are not feasible if going beyond max_charge_percentage. 1 = feasible, 0 = infeasible
        feasible_discharge = {lvl: 0 for lvl in action_levels}                               # Initialises the discharging feasibility dictionary for each action level. Discharge action levels are not feasible if going below min_charge_percentage. 1 = feasible, 0 = infeasible

        upcoming_trip_distance = np.nan                                                      # Initialises the upcoming trip distance dictionary
        upcoming_trip_energy_kWh = np.nan                                                    # Initialises the upcoming trip energy dictionary
        upcoming_energy_requirement_percent = np.nan                                         # Initialises the upcoming energy requirement (in % SoC) dictionary
        battery_energy_level_current_percent = np.nan                                        # Initialises the current battery energy level (in % SoC) dictionary
        energy_gap = np.nan                                                                  # Initialises the difference between the upcoming trip energy requirement and current SoC (in % SoC) dictionary

        target_energy_kWh = np.nan                                                           # Initialises the target energy level (in kWh) dictionary
        total_charge_session_kWh = np.nan                                                    # Initialises the total charge session energy level (in kWh) dictionary 
        total_discharge_session_kWh = np.nan                                                 # Initialises the total discharge session energy level (in kWh) dictionary 

        price_utility_scaled = np.nan                                                          # Initialises a dictionary for the base measure of current energy price from which charge and discharge price utilities are calculated
        charge_utility_price = np.nan                                                        # Initialises the price component of charge utility dictionary
        discharge_utility_price = np.nan                                                     # Initialises the price component of discharge utility dictionary

        action = "none"                                                                      # Initialises the selected action (charge action, discharge action, drive or none) dictionary
        amount_charged_during_timestep_kWh = 0.0                                             # Initialises a dictionary for the amount of energy charged (if positive) or discharged (if negative) in kWh, per timestep
        is_new_decision = 0                                                                  # Initialises a dictionary, tracking if each timestep involves a new decision being taken. 1 = a new decision, 0 = not a new decsion 

        
# -------------------------
# 8. Determining the effects of actions at different locations
# -------------------------

# 8.1 Determining upcoming trip distance and upcoming trip energy
        
        if location == "home":
            upcoming_trip_distance = trip_distance_out                                       # Trip distance out (from home to work, in miles) is taken, per individual, from the NTS dataset
            upcoming_trip_energy_kWh = energy_req_home_to_work_kWh                           # Defined in section 3.4
        elif location == "work" and timestep < departure_timestep_work:
            upcoming_trip_distance = trip_distance_return                                    # Return trip distance (from work to home, in miles) is taken, per individual, from the NTS dataset
            upcoming_trip_energy_kWh = energy_req_work_to_home_kWh                           # Defined in section 3.4

        if location not in ["home", "work"]:                                                 # Resets persistent actions if not at home or work
            action_duration = 0
            current_action = None
            charge_session_remaining_kWh = 0.0

# 8.2 Changing battery energy level, whilst driving

        if location == "driving_out":                                                                                                    # See section 5
            battery_energy_level_current_kWh = max(0, battery_energy_level_current_kWh - driving_consumption_per_timestep_out_kWh)       # Reduces battery energy level each time step, whilst driving. Ensures a negative value can't be returned
            action = "drive"                                                                                                             # Whilst location is 'driving_out', the action is 'drive'

        elif location == "driving_return":                                                                                               # See section 5
            battery_energy_level_current_kWh = max(0, battery_energy_level_current_kWh - driving_consumption_per_timestep_return_kWh)    # Reduces battery energy level each time step, whilst driving. Ensures a negative value can't be returned
            action = "drive"                                                                                                             # Whilst location is 'driving_return', the action is 'drive'

# 8.3 Setting home and work target charge levels and charge rates
        
        elif location in ["home", "work"]:                                                                                                             # See section 5
            target_energy_kWh = target_charge_level(timestep, location)                                                                                # Target energy level (kWh), by time step. Defined by the function in section 6
            charging_rate_kWh_per_timestep = home_charging_rate_kWh_per_timestep if location == "home" else work_charging_rate_kWh_per_timestep        # Charging rates (in kW/h) defined in section 3.2

# 8.4 Energy level and price utility function components

            upcoming_energy_requirement_percent = target_energy_kWh / battery_capacity_kWh                                   # Target energy level, expressed as % SoC
            battery_energy_level_current_percent = battery_energy_level_current_kWh / battery_capacity_kWh                   # Current energy level, expressed as % SoC
            energy_gap = upcoming_energy_requirement_percent - battery_energy_level_current_percent                          # Difference between current and target energy levels (in % SoC)

            E_req = target_energy_kWh                                                                                        # Simplification of target energy, current energy level and battery capacity variable names
            E_now = battery_energy_level_current_kWh
            C = battery_capacity_kWh

            if use_price_utility:                                                                                            # If price utility is included in the utility function (see section 1), price utilities are calculated as follows:
                price_utility_scaled = (1 - ((price_at_timestep_t - (base_price - amplitude)) * 100 * 0.05))                 # Converts price to a scale of 1 (for lowest price) to -1 (for highest price)
                charge_utility_price = price_utility_scaled                                                                  # The price based component of charging utility = beta2 (representing each agent's price sensitivity) * scaled price utility
                discharge_utility_price = -price_utility_scaled                                                              # The price based component of discharging utility = beta2 (representing each agent's price sensitivity) * - scaled price utility
            else:
                price_utility_scaled = 0.0                                                                                   # If price isn't included in the utility function, price utilities are set to zero
                charge_utility_price = 0.0
                discharge_utility_price = 0.0

# 8.5 MNL-based utility and probability calculations for 'charge', 'discharge' and 'none' actions

            if action_duration > 0:                                                                                          # The following steps determine updates when an action is continued from the previous timestep
                action = current_action                                                                                      # curent_action defined in section 7 and 8.1
                amount_charged_during_timestep_kWh = min(charging_rate_kWh_per_timestep, charge_session_remaining_kWh)       # The amount charged during a timestep equals the charge rate (kWh) or less, if less than the charge rate is remaining to charge during an ongoing charging action
                charge_session_remaining_kWh -= amount_charged_during_timestep_kWh                                           # Reduces the amount remaining to be charged, following amounts charged during each timestep
                action_duration -= 1                                                                                         # Reduces the action duration by 1 timestep, following completion of each timestep

                if current_action and current_action.startswith("charge_"):
                    total_charge_session_kWh = current_total_charge_kWh                                                      # The current total amount to charge is set as equal to the total amount to charge for the session
                elif current_action and current_action.startswith("discharge_"):
                    total_discharge_session_kWh = current_total_discharge_kWh                                                # The current total amount to discharge is set as equal to the total amount to discharge for the session

            else:
                is_new_decision = 1                                                                                          # When a charging or discharging action ends, is_new_decision is set to 1, meaning a new action must be selected
                Utility_none = beta0_none                                                                                    # The utility of deciding to do nothing (not charge or discharge) is defined by beta0_none (see section 4.1)

                feasible_actions = ["none"]                                                                                  # Creates a list of feasible actions (charge actions that will not go above the max charge level or below the min charge level
                feasible_action_utilities = [Utility_none]                                                                              # Creates a list of utilities of feasible actions

                for lvl in action_levels:                                                                                    # For loop iterates over charging and discharging actions and evaluates the utility of each one
                    # CHARGE option
                    E_post_c = E_now + lvl * C                                                                               # Energy level % post charge for each charging action
                    can_charge = (E_post_c <= C) and (E_post_c <= max_charge_kWh)                                            # Determines the feasibility of each charge action (ensuring it will not exceed the battery capacity or max charge level), returning 1 if an action is feasible and 0 if it is not
                    feasible_charge[lvl] = int(can_charge)                                                                   # Converts can_charge to an integer and stores it in the feasible charge disctionary, at the key lvl. This resets each time step, defining the feasible actions for that time step

                    charge_utility_energy_level, gap_c_signed, dist_c = piecewise_energy_term(E_post_c, E_req, C, k_shortfall=15.0, k_surplus=8.0)                        # The piecewise energy term (defined in section 2) returns energy level related charging utility, plus the signed gap between post charge energy and target energy, and the absolute (unsigned) difference between post charge energy and target energy
                    Uc = (beta0_charge + (beta1_charge * charge_utility_energy_level) + (beta2_charge * charge_utility_price)) * 10                                       # The charge action utility function

                    charge_utility_energy_level_by_level[lvl] = charge_utility_energy_level                                  # The energy level component of charge utility is stored in the charge_utility_energy_level_by_level dictionary for each charge level
                    distance_after_charge[lvl] = dist_c                                                                      # Absolute distance from target energy post-charge is stored in the distance_after_charge dictionary
                    shortfall_after_charge[lvl] = max(0.0, (E_req - E_post_c) / C)                                           # Amount of energy short of the target is stored in the distance_after_charge dictionary
                    utility_charge[lvl] = Uc if can_charge else np.nan                                                       # Utility for feasible charge levels is returned. If infeasible, np.nan is returned

                    if can_charge:
                        feasible_actions.append(f"charge_{int(lvl * 100)}")                                                  # Feasible actions are added to the feasible_actions list, in the form of, for example, 'charge_10'
                        feasible_action_utilities.append(Uc)                                                                 # Corresponding utilities for feasible actions are added to the feasible_action_utilities list

                    # DISCHARGE option
                    E_post_d = E_now - lvl * C                                                                               # Energy level % post discharge for each discharging action
                    can_discharge_abs = (E_post_d >= 0.0)
                    can_discharge_beh = (E_post_d >= min_charge_kWh)
                    can_discharge = can_discharge_abs and can_discharge_beh                                                  # Determines the feasibility of each discharge action (ensuring it will not be below 0% charge or the behavioural constraint of the min charge level), returning 1 if an action is feasible and 0 if it is not
                    feasible_discharge[lvl] = int(can_discharge)                                                             # Converts can_discharge to an integer and stores it in the feasible discharge disctionary, at the key lvl. This resets each time step, defining the feasible actions for that time step

                    discharge_utility_energy_level, gap_d_signed, dist_d = piecewise_energy_term(E_post_d, E_req, C, k_shortfall=15.0, k_surplus=8.0)                     # The piecewise energy term (defined in section 2) returns energy level related discharging utility, plus the signed gap between post discharge energy and target energy, and the absolute (unsigned) difference between post discharge energy and target energy
                    Ud = (beta0_discharge + (beta1_discharge * discharge_utility_energy_level) + (beta2_discharge * discharge_utility_price)) * 10                        # The discharge action utility function

                    discharge_utility_energy_level_by_level[lvl] = discharge_utility_energy_level                            # The energy level component of discharge utility is stored in the discharge_utility_energy_level_by_level dictionary for each charge level
                    distance_after_discharge[lvl] = dist_d                                                                   # Absolute distance from target energy post-discharge is stored in the distance_after_discharge dictionary
                    surplus_after_discharge[lvl] = max(0.0, (E_post_d - E_req) / C)                                          # Amount of energy above the target is stored in the distance_after_charge dictionary
                    utility_discharge[lvl] = Ud if can_discharge else np.nan                                                 # Utility for feasible discharge levels is returned. If infeasible, np.nan is returned

                    if can_discharge:
                        feasible_actions.append(f"discharge_{int(lvl * 100)}")                                               # Feasible actions are added to the feasible_actions list, in the form of, for example, 'discharge_10'
                        feasible_action_utilities.append(Ud)                                                                 # Corresponding utilities for feasible actions are added to the feasible_action_utilities list

                u = np.array(feasible_action_utilities, dtype=float)                                                         # Utility values for all possible actions are converted into a NumPy array of floats, ready for mathematical operations 
                u = u - np.max(u)                                                                                            # Subtracts the maximum utility value from each utility, provides stability by avoiding very large numbers when exponentiating in the MNL formula
                probs = np.exp(u) / np.sum(np.exp(u))                                                                        # Softmax function, calculating the probability of selecting each charging option, in line with the MNL formula
                prob_map = {a: probs[i] for i, a in enumerate(feasible_actions)}                                             # Creates a dictionary assigning a probability to each action name 

                action = np.random.choice(feasible_actions, p=probs)                                                         # An action is selected randomly across the MNL probability distribution 

                for lvl in action_levels:
                    prob_charge[lvl] = prob_map.get(f"charge_{int(lvl * 100)}", 0.0)                                         # Charge and discharge probabilities are extracted for each action level, including assigning 0 for infeasible actions (using the .get(key, 0.0) method
                    prob_discharge[lvl] = prob_map.get(f"discharge_{int(lvl * 100)}", 0.0)                                   # prob_charge and prob_discharge dictionaries are returned

                P_none = prob_map.get("none", 0.0)                                                                           # Returns the probability of doing nothing for this timestep
                P_charge = sum(prob_charge[lvl] for lvl in action_levels)                                                    # Returns the probability of completing any charging action
                P_discharge = sum(prob_discharge[lvl] for lvl in action_levels)                                              # Returns the probability of completing any discharging action

# 8.6 Implementation of charging and discharging actions
                
                if action.startswith("charge_"):
                    lvl = int(action.split("_")[1]) / 100.0                                                                  # Extracts the charge level from the action string
                    total_charge_session_kWh = min(lvl * C, max_charge_kWh - E_now, C - E_now)                               # Calculates total charge session energy, ensuring that max charge level is not exceeded
                    total_charge_session_kWh = max(0.0, total_charge_session_kWh)                                            # Ensures the charge amount is positive

                    duration = math.ceil(total_charge_session_kWh / charging_rate_kWh_per_timestep) if total_charge_session_kWh > 0 else 0                 # Calculates charge session duration in timesteps
                    if location == "home":
                        remaining_time_home = departure_timestep_home - timestep if timestep < departure_timestep_home else (timesteps - 1) - timestep     # Ensures charging cannot go beyond the next required home or work departure time
                        duration = min(duration, remaining_time_home)
                    else:
                        duration = min(duration, departure_timestep_work - timestep)

                    action_duration = max(duration - 1, 0)                                                                   # Initialises the charging session - setting up all tracking variables for the charging session 
                    current_action = action
                    charge_session_remaining_kWh = total_charge_session_kWh
                    amount_charged_during_timestep_kWh = min(charging_rate_kWh_per_timestep, charge_session_remaining_kWh)
                    charge_session_remaining_kWh -= amount_charged_during_timestep_kWh
                    current_total_charge_kWh = total_charge_session_kWh

                elif action.startswith("discharge_"):
                    lvl = int(action.split("_")[1]) / 100.0                                                                  # Extracts the discharge level from the action string
                    total_discharge_session_kWh = min(lvl * C, E_now - min_charge_kWh, E_now)                                # Calculates total discharge session energy, ensuring this does not go below the min charge level
                    total_discharge_session_kWh = max(0.0, total_discharge_session_kWh)                                      # Ensures the discharge amount is positive

                    duration = math.ceil(total_discharge_session_kWh / charging_rate_kWh_per_timestep) if total_discharge_session_kWh > 0 else 0           # Calculates discharge session duration in timesteps
                    if location == "home":
                        remaining_time_home = departure_timestep_home - timestep if timestep < departure_timestep_home else (timesteps - 1) - timestep     # Ensures charging cannot go beyond the next required home or work departure time
                        duration = min(duration, remaining_time_home)
                    else:
                        duration = min(duration, departure_timestep_work - timestep)

                    action_duration = max(duration - 1, 0)                                                                   # Initialises the charging session - setting up all tracking variables for the charging session 
                    current_action = action
                    charge_session_remaining_kWh = total_discharge_session_kWh
                    amount_charged_during_timestep_kWh = min(charging_rate_kWh_per_timestep, charge_session_remaining_kWh)
                    charge_session_remaining_kWh -= amount_charged_during_timestep_kWh
                    current_total_discharge_kWh = total_discharge_session_kWh

                else:
                    action_duration = 0                                                                                      # If neither charge, nor discharge is selected                                                                                     
                    current_action = None
                    charge_session_remaining_kWh = 0.0

        if action.startswith("charge_"):                                                                                                                    # Updates battery energy level (kWh) and accumulated profit levels following each charging or discharging action
            battery_energy_level_current_kWh = min(max_charge_kWh, battery_energy_level_current_kWh + amount_charged_during_timestep_kWh)
            accumulated_profit -= price_at_timestep_t * amount_charged_during_timestep_kWh 
        elif action.startswith("discharge_"):
            battery_energy_level_current_kWh = max(min_charge_kWh, battery_energy_level_current_kWh - amount_charged_during_timestep_kWh)
            accumulated_profit += price_at_timestep_t * amount_charged_during_timestep_kWh

# -------------------------
# 9. Simulation dataframe, per individual
# -------------------------
        row = {
            "IndividualID": individual_id,
            "Age_Group": age_group,
            "Gender": gender,
            "Segment": segment,
            "Battery_Capacity_kWh": battery_capacity_kWh,
            "Home_Charger_kW": home_charger_kw,
            "Work_Charger_kW": work_charger_kw,
            "Initial_Energy_kWh": initial_energy_kWh,
            "Timestep": timestep,
            "Time": time_str,
            "Location": location,
            "Battery_Energy_Level_kWh": battery_energy_level_current_kWh,
            "Battery_Energy_Level_%": battery_energy_level_current_kWh / battery_capacity_kWh,
            "Trip_Start_Time_Out_Mins": trip_start_out_mins,
            "Trip_End_Time_Out_Mins": trip_end_out_mins,
            "Trip_Start_Time_Return_Mins": trip_start_return_mins,
            "Trip_End_Time_Return_Mins": trip_end_return_mins,
            "Upcoming_Trip_Distance_Miles": upcoming_trip_distance,
            "Average_Speed_Out_mph": avg_speed_out,
            "Average_Speed_Return_mph": avg_speed_return,
            "Upcoming_Trip_Energy_kWh": upcoming_trip_energy_kWh,
            "Upcoming_Energy_Requirement_kWh": target_energy_kWh if location in ["home", "work"] else np.nan,
            "Upcoming_Energy_Requirement_%": upcoming_energy_requirement_percent,
            "Energy_Gap_%": energy_gap,
            "Energy_Price_Pounds": price_at_timestep_t,
            "Action": action,
            "Total_Charge_kWh": total_charge_session_kWh,
            "Total_Discharge_kWh": total_discharge_session_kWh,
            "Action_Duration_Timesteps": action_duration,
            "Amount_Charged_During_Timestep_kWh": -amount_charged_during_timestep_kWh if action.startswith("discharge_") else amount_charged_during_timestep_kWh,
            "Charge_Session_Remaining_kWh": charge_session_remaining_kWh,
            "Accumulated_Charging_Profit_Pounds": accumulated_profit,
            "Beta0_Charge": beta0_charge,
            "Beta1_Charge": beta1_charge,
            "Beta2_Charge": beta2_charge,
            "Beta0_Discharge": beta0_discharge,
            "Beta1_Discharge": beta1_discharge,
            "Beta2_Discharge": beta2_discharge,
            "Price_Utility_Scaled": price_utility_scaled,
            "Charge_Utility_Price": charge_utility_price,
            "Discharge_Utility_Price": discharge_utility_price,
            "Utility_None": Utility_none,
            "Probability_None": P_none,
            "Probability_Charge": P_charge,
            "Probability_Discharge": P_discharge,
            "Is_New_Decision": is_new_decision
        }

        for lvl in action_levels:
            p = int(lvl * 100)
            row[f"Utility_Charge_{p}"] = utility_charge[lvl]
            row[f"Utility_Discharge_{p}"] = utility_discharge[lvl]
            row[f"Probability_Charge_{p}"] = prob_charge[lvl]
            row[f"Probability_Discharge_{p}"] = prob_discharge[lvl]
            row[f"Distance_After_Charge_{p}"] = distance_after_charge[lvl]
            row[f"Distance_After_Discharge_{p}"] = distance_after_discharge[lvl]
            row[f"Shortfall_After_Charge_{p}"] = shortfall_after_charge[lvl]
            row[f"Surplus_After_Discharge_{p}"] = surplus_after_discharge[lvl]
            row[f"Feasible_Charge_{p}"] = feasible_charge[lvl]
            row[f"Feasible_Discharge_{p}"] = feasible_discharge[lvl]
            row[f"Charge_Utility_Energy_Level_{p}"] = charge_utility_energy_level_by_level[lvl]
            row[f"Discharge_Utility_Energy_Level_{p}"] = discharge_utility_energy_level_by_level[lvl]

        output.append(row)

    df_sim = pd.DataFrame(output)                                                                                                      # This is the output dataframe per individual, comprising the above fields
    numeric_columns = [c for c in df_sim.columns if c not in ['Age_Group', 'Gender', 'Segment', 'Location', 'Action', 'Time']]         # All but three fields are non-numeric
    df_sim[numeric_columns] = df_sim[numeric_columns].astype(float)                                                                    # Ensures all numeric fields are floats
    return df_sim


# -------------------------
# 10. Beta values for each population segment.
# -------------------------

# beta1_charge and beta1_discharge define EV user attitudes in the MNL utility function towards energy level, when deciding whether to charge, discharge or do nothing
# beta2_charge and beta2_discharge define EV user attitudes in the MNL utility function energy price, when deciding whether to charge, discharge or do nothing

segment_parameters = {
    "Female 17-20": {"beta1_charge": 0.85, "beta2_charge": 0.3, "beta1_discharge": 0.85, "beta2_discharge": 0.3},
    "Male 17-20": {"beta1_charge": 0.9, "beta2_charge": 0.3, "beta1_discharge": 0.9, "beta2_discharge": 0.3},
    "Female 21-29": {"beta1_charge": 0.85, "beta2_charge": 0.3, "beta1_discharge": 0.85, "beta2_discharge": 0.3},
    "Male 21-29": {"beta1_charge": 0.9, "beta2_charge": 0.3, "beta1_discharge": 0.9, "beta2_discharge": 0.3},
    "Female 30-39": {"beta1_charge": 0.75, "beta2_charge": 0.15, "beta1_discharge": 0.75, "beta2_discharge": 0.15},
    "Male 30-39": {"beta1_charge": 0.8, "beta2_charge": 0.15, "beta1_discharge": 0.8, "beta2_discharge": 0.15},
    "Female 40-49": {"beta1_charge": 0.75, "beta2_charge": 0.15, "beta1_discharge": 0.75, "beta2_discharge": 0.15},
    "Male 40-49": {"beta1_charge": 0.8, "beta2_charge": 0.15, "beta1_discharge": 0.8, "beta2_discharge": 0.15},
    "Female 50-59": {"beta1_charge": 0.65, "beta2_charge": 0.3, "beta1_discharge": 0.65, "beta2_discharge": 0.3},
    "Male 50-59": {"beta1_charge": 0.7, "beta2_charge": 0.3, "beta1_discharge": 0.7, "beta2_discharge": 0.3},
    "Female 60+": {"beta1_charge": 0.65, "beta2_charge": 0.3, "beta1_discharge": 0.65, "beta2_discharge": 0.3},
    "Male 60+": {"beta1_charge": 0.7, "beta2_charge": 0.3, "beta1_discharge": 0.7, "beta2_discharge": 0.3}
}


# -------------------------
# 11. Create combined dataframe
# -------------------------

# 11.1 Pull data from the NTS dataset

trip_df = pd.read_csv("NTS_Sampled_500_Individuals_Weekdays_All_Segments_Final.csv")           # Pulls in data from the National Travel Survey 2023 csv file
all_output = []                                                                                # Creates an empty list for output for all individuals combined (with one df_individual returned per individual)

for idx, row in trip_df.iterrows():                                                            # Iterates over the dataframe rows, returning row indices (idx) and row data (row) as series
    segment = row["Segment"]                                                                   # Reads the segment label for each row
    parameters = segment_parameters.get(segment)                                               # Returns the parameter dictionary for the key returned by segment
    if parameters is None:
        raise ValueError(f"Unknown segment '{segment}' for individual {idx}.")                 # Returns an error if the segment name doesn't appear in the segment_parameters dictionary

# 11.2 A dictionary of betas is created for each individual, pulling in parameter values from the segment_parameters dictionary
    
    betas = {
        "beta0_charge": 0.0,
        "beta1_charge": parameters["beta1_charge"],
        "beta2_charge": parameters["beta2_charge"],
        "beta0_discharge": 0.0,
        "beta1_discharge": parameters["beta1_discharge"],
        "beta2_discharge": parameters["beta2_discharge"]
    }

# 11.3 Determine departure and arrival timesteps for individuals 

# Departure and arrival minutes are taken from the NTS dataset and converted to 15-minute timesteps
    
    departure_timestep_home = int(round(row["TripStart_out"] / 15))
    arrival_timestep_work = int(round(row["TripEnd_out"] / 15))
    departure_timestep_work = int(round(row["TripStart_return"] / 15))
    arrival_timestep_home = int(round(row["TripEnd_return"] / 15))

# Trip distances (miles) and average speeds (miles per hour) are taken from the NTS dataset
    
    trip_distance_out = row["TripDisExSW_out"]
    trip_distance_return = row["TripDisExSW_return"]
    avg_speed_out = row["Average_Speed_mph_out"]
    avg_speed_return = row["Average_Speed_mph_return"]

# 11.4 Create dataframe of simulation parameters for each individual
    
    df_individual = simulation_parameters_for_each_individual(
        base_price=base_price,                                    # Defined in section 1
        amplitude=amplitude,                                      # Defined in section 1
        betas=betas,                                              # Defined in section 11.2
        individual_id=row["IndividualID"],
        age_group=row["Age_Group"],
        gender=row["Gender"],
        segment=row["Segment"],
        departure_timestep_home=departure_timestep_home,          # Defined in section 11.3
        arrival_timestep_work=arrival_timestep_work,              # Defined in section 11.3
        departure_timestep_work=departure_timestep_work,          # Defined in section 11.3
        arrival_timestep_home=arrival_timestep_home,              # Defined in section 11.3
        trip_distance_out=trip_distance_out,                      # Defined in section 11.3
        trip_distance_return=trip_distance_return,                # Defined in section 11.3
        avg_speed_out=avg_speed_out,                              # Defined in section 11.3
        avg_speed_return=avg_speed_return,                        # Defined in section 11.3
        trip_start_out_mins=row["TripStart_out"],
        trip_end_out_mins=row["TripEnd_out"],
        trip_start_return_mins=row["TripStart_return"],
        trip_end_return_mins=row["TripEnd_return"]
    )

    all_output.append(df_individual)                              # all_output is a list of per-individual dataframes

df_all = pd.concat(all_output, ignore_index=True)                 # df_all is a single combined dataframe for all individuals, across all timesteps

# -------------------------
# 12. Export CSV
# -------------------------

df_all.to_csv(
    "multi_individual_simulation_output.csv",
    index=False,
    float_format="%.2f",
    quoting=csv.QUOTE_MINIMAL
)

print("Simulation complete. Output saved to multi_individual_simulation_output.csv.")

Simulation complete. Output saved to multi_individual_simulation_output.csv.
